# 🍟 Snackspert - Instagram Recensies naar Google Docs

Dit notebook verzamelt alle recensies van **instagram.com/snackspert** en maakt per recensie een apart Google Docs-bestand aan in een Google Drive-map.

## Hoe te gebruiken
1. Voer elke cel uit door op het **▶ play-knopje** te klikken (of druk `Shift+Enter`)
2. Bij stap 2 moet je inloggen met je Google-account
3. Bij stap 3 vul je de instellingen in
4. Stap 4 en 5 doen het werk!

---

## Stap 1: Installeer benodigde packages
Dit hoef je maar één keer te doen per sessie.

In [ ]:
!pip install -q instaloader google-api-python-client google-auth-httplib2 google-auth-oauthlib
print("\n\u2705 Alle packages ge\u00efnstalleerd!")

## Stap 2: Log in met je Google-account
Er verschijnt een pop-up om in te loggen. Dit geeft het notebook toegang tot je Google Drive en Google Docs.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.auth import default
creds, _ = default()

from googleapiclient.discovery import build
docs_service = build('docs', 'v1', credentials=creds)
drive_service = build('drive', 'v3', credentials=creds)

print("\u2705 Ingelogd en verbonden met Google Drive & Docs!")

## Stap 3: Instellingen
Pas hier de instellingen aan als dat nodig is.

In [ ]:
# === INSTELLINGEN ===

# Het Instagram-account om te scrapen
INSTAGRAM_ACCOUNT = "snackspert"

# Maximaal aantal recensies ophalen (0 = alles)
MAX_RECENSIES = 0

# Naam van de Google Drive-map waar de documenten in komen
# (wordt automatisch aangemaakt als deze niet bestaat)
DRIVE_MAP_NAAM = "Snackspert Recensies"

# === Optioneel: Instagram login (alleen nodig als het profiel priv\u00e9 is) ===
INSTAGRAM_LOGIN_GEBRUIKER = ""  # Laat leeg als niet nodig
INSTAGRAM_LOGIN_WACHTWOORD = ""  # Laat leeg als niet nodig

print(f"\u2705 Instellingen geladen:")
print(f"   Account: @{INSTAGRAM_ACCOUNT}")
print(f"   Max recensies: {'alles' if MAX_RECENSIES == 0 else MAX_RECENSIES}")
print(f"   Drive-map: {DRIVE_MAP_NAAM}")

## Stap 4: Recensies ophalen van Instagram
Dit kan even duren, afhankelijk van het aantal posts.

In [ ]:
import re
from dataclasses import dataclass, field
from datetime import datetime
import instaloader


@dataclass
class Review:
    """Een enkele Instagram-recensie."""
    post_id: str
    date: datetime
    caption: str
    likes: int
    comments_count: int
    image_url: str
    permalink: str
    hashtags: list = field(default_factory=list)
    location: str = None

    @property
    def title(self) -> str:
        if not self.caption:
            return f"Recensie {self.date.strftime('%d-%m-%Y')}"
        first_line = self.caption.split('\n')[0].strip()
        clean = re.sub(r'[^\w\s\-&\'(),.]', '', first_line).strip()
        if len(clean) > 80:
            clean = clean[:77] + '...'
        return clean or f"Recensie {self.date.strftime('%d-%m-%Y')}"


# --- Scraper starten ---
print(f"\U0001f4f8 Recensies ophalen van @{INSTAGRAM_ACCOUNT}...\n")

loader = instaloader.Instaloader(
    download_pictures=False,
    download_videos=False,
    download_video_thumbnails=False,
    download_geotags=False,
    download_comments=False,
    save_metadata=False,
    compress_json=False,
)

# Optioneel inloggen
if INSTAGRAM_LOGIN_GEBRUIKER and INSTAGRAM_LOGIN_WACHTWOORD:
    try:
        loader.login(INSTAGRAM_LOGIN_GEBRUIKER, INSTAGRAM_LOGIN_WACHTWOORD)
        print(f"\u2705 Ingelogd als {INSTAGRAM_LOGIN_GEBRUIKER}")
    except Exception as e:
        print(f"\u26a0\ufe0f Kon niet inloggen: {e}")
        print("   Ga verder zonder login...")

# Profiel ophalen
try:
    profile = instaloader.Profile.from_username(loader.context, INSTAGRAM_ACCOUNT)
    print(f"\u2705 Profiel gevonden: @{profile.username} ({profile.mediacount} posts)\n")
except Exception as e:
    raise SystemExit(f"\u274c Profiel @{INSTAGRAM_ACCOUNT} niet gevonden: {e}")

# Posts ophalen
recensies = []
for i, post in enumerate(profile.get_posts()):
    if MAX_RECENSIES > 0 and i >= MAX_RECENSIES:
        break

    caption = post.caption or ""
    hashtags = re.findall(r'#(\w+)', caption)

    review = Review(
        post_id=post.shortcode,
        date=post.date_utc,
        caption=caption,
        likes=post.likes,
        comments_count=post.comments,
        image_url=post.url,
        permalink=f"https://www.instagram.com/p/{post.shortcode}/",
        hashtags=hashtags,
        location=post.location.name if post.location else None,
    )
    recensies.append(review)
    print(f"  [{i + 1}] {review.title[:60]}")

print(f"\n\u2705 {len(recensies)} recensies opgehaald!")

## Stap 5: Recensies opslaan als Google Docs
Per recensie wordt een apart document aangemaakt in de map op je Google Drive.

In [ ]:
# --- Google Drive-map zoeken of aanmaken ---
print(f"\U0001f4c1 Map '{DRIVE_MAP_NAAM}' zoeken of aanmaken...\n")

# Zoek of de map al bestaat
query = f"name = '{DRIVE_MAP_NAAM}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
result = drive_service.files().list(q=query, fields='files(id, name)').execute()
folders = result.get('files', [])

if folders:
    folder_id = folders[0]['id']
    print(f"\u2705 Bestaande map gevonden: {DRIVE_MAP_NAAM}")
else:
    folder_metadata = {
        'name': DRIVE_MAP_NAAM,
        'mimeType': 'application/vnd.google-apps.folder'
    }
    folder = drive_service.files().create(body=folder_metadata, fields='id').execute()
    folder_id = folder['id']
    print(f"\u2705 Nieuwe map aangemaakt: {DRIVE_MAP_NAAM}")

print(f"   Map-ID: {folder_id}")
print(f"   \U0001f517 https://drive.google.com/drive/folders/{folder_id}\n")


# --- Functie om een document aan te maken ---
def maak_recensie_doc(review, folder_id):
    """Maak een Google Docs-bestand aan voor \u00e9\u00e9n recensie."""
    doc_title = f"Snackspert - {review.title}"

    # Leeg document aanmaken
    doc = docs_service.documents().create(body={'title': doc_title}).execute()
    doc_id = doc['documentId']

    # Verplaats naar de juiste map
    drive_service.files().update(
        fileId=doc_id,
        addParents=folder_id,
        removeParents='root',
        fields='id, parents'
    ).execute()

    # Document vullen met inhoud
    sections = []

    # Titel
    sections.append({'text': f"{review.title}\n", 'style': 'HEADING_1'})

    # Metadata
    meta_lines = [
        f"Datum: {review.date.strftime('%d %B %Y')}",
        f"Likes: {review.likes}",
        f"Reacties: {review.comments_count}",
    ]
    if review.location:
        meta_lines.append(f"Locatie: {review.location}")
    meta_lines.append(f"Instagram: {review.permalink}")
    sections.append({'text': '\n'.join(meta_lines) + '\n\n', 'style': 'NORMAL_TEXT'})

    # Recensie tekst
    sections.append({'text': 'Recensie\n', 'style': 'HEADING_2'})
    sections.append({'text': (review.caption or '(Geen tekst)') + '\n\n', 'style': 'NORMAL_TEXT'})

    # Hashtags
    if review.hashtags:
        sections.append({'text': 'Hashtags\n', 'style': 'HEADING_2'})
        sections.append({'text': ' '.join(f'#{tag}' for tag in review.hashtags) + '\n', 'style': 'NORMAL_TEXT'})

    # Afbeelding link
    sections.append({'text': '\nAfbeelding\n', 'style': 'HEADING_2'})
    sections.append({'text': review.image_url + '\n', 'style': 'NORMAL_TEXT'})

    # Requests opbouwen
    requests = []
    index = 1
    for section in sections:
        text = section['text']
        requests.append({
            'insertText': {
                'location': {'index': index},
                'text': text,
            }
        })
        if section['style'] != 'NORMAL_TEXT':
            requests.append({
                'updateParagraphStyle': {
                    'range': {'startIndex': index, 'endIndex': index + len(text)},
                    'paragraphStyle': {'namedStyleType': section['style']},
                    'fields': 'namedStyleType',
                }
            })
        index += len(text)

    if requests:
        docs_service.documents().batchUpdate(
            documentId=doc_id, body={'requests': requests}
        ).execute()

    return f"https://docs.google.com/document/d/{doc_id}/edit"


# --- Alle recensies verwerken ---
print(f"\U0001f4dd {len(recensies)} documenten aanmaken...\n")

resultaten = []
for i, review in enumerate(recensies, 1):
    print(f"  [{i}/{len(recensies)}] {review.title[:50]}...", end=" ")
    try:
        url = maak_recensie_doc(review, folder_id)
        resultaten.append({'title': review.title, 'url': url})
        print(f"\u2705")
    except Exception as e:
        resultaten.append({'title': review.title, 'url': None, 'error': str(e)})
        print(f"\u274c {e}")

# Samenvatting
gelukt = [r for r in resultaten if r.get('url')]
mislukt = [r for r in resultaten if not r.get('url')]

print(f"\n{'='*50}")
print(f"\u2705 KLAAR!")
print(f"{'='*50}")
print(f"  Totaal:    {len(recensies)} recensies")
print(f"  Gelukt:    {len(gelukt)}")
if mislukt:
    print(f"  Mislukt:   {len(mislukt)}")
print(f"\n\U0001f4c2 Open je Google Drive-map:")
print(f"   https://drive.google.com/drive/folders/{folder_id}")

## Stap 6 (optioneel): Bekijk een overzicht
Bekijk een tabel met alle aangemaakte documenten.

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        'Titel': r['title'][:50],
        'Status': '\u2705' if r.get('url') else '\u274c',
        'Google Docs Link': r.get('url', r.get('error', '-'))
    }
    for r in resultaten
])

print(f"\U0001f4ca Overzicht van alle {len(resultaten)} recensies:\n")
df